# LEAA Training — Stage 3: `static_far`
**Target accuracy:** 85%  |  **Timesteps:** 15,000,000

### Setup Instructions
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Add your GitHub PAT to Colab Secrets:
   - Left sidebar → 🔑 Secrets → `GITHUB_TOKEN`
3. Run all cells in order
4. When session expires, re-open this notebook and run all cells — it auto-resumes


In [ ]:
# Cell 1: Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    raise RuntimeError('No GPU detected! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# Cell 2: Authenticate & Clone repo
from google.colab import userdata
import os, subprocess

TOKEN = userdata.get('GITHUB_TOKEN')
REPO = 'Sathvik-Chowdary-Veerapaneni/Language-Embeded-Agent-Action'
CLONE_URL = f'https://{{TOKEN}}@github.com/{{REPO}}.git'

if not os.path.exists('/content/leaa'):
    subprocess.run(['git', 'clone', CLONE_URL, '/content/leaa'], check=True)
else:
    subprocess.run(['git', 'pull'], cwd='/content/leaa', check=True)

# Configure git identity for pushes
subprocess.run(['git', 'config', 'user.email', 'colab@leaa.bot'], cwd='/content/leaa')
subprocess.run(['git', 'config', 'user.name', 'Colab Training Bot'], cwd='/content/leaa')

# Embed token in remote URL so pushes work without interactive auth
subprocess.run(['git', 'remote', 'set-url', 'origin', CLONE_URL], cwd='/content/leaa')
print('✓ Repo ready at /content/leaa')

In [ ]:
# Cell 3: Install dependencies
%cd /content/leaa
!pip install -q -r requirements.txt
print('✓ Dependencies installed')

In [ ]:
# Cell 4: Run training
# This cell runs for the full session (~11 hrs).
# Checkpoints auto-sync to GitHub every 30 min.
# If session expires, re-run all cells — training resumes from last checkpoint.
%cd /content/leaa
!python scripts/colab_train.py --stage 3 --timesteps 15000000 --num-envs 4

In [ ]:
# Cell 5: (Optional) Evaluate this stage after training
%cd /content/leaa
!python rl_training/evaluate.py \\
    --model rl_training/checkpoints/static_far_best.zip \\
    --vecnorm rl_training/checkpoints/vecnormalize_static_far_best.pkl \\
    --stage static_far \\
    --episodes 200